# Notebook 05: DataLoader and Dataset

## Why This Matters

Data loading is almost always the GPU utilisation bottleneck in production training pipelines. Engineers who do not understand `num_workers`, `pin_memory`, and `persistent_workers` routinely achieve 30-60% GPU utilisation when 95%+ is possible. Beyond performance, incorrect `IterableDataset` implementations cause silent data duplication or data loss when using multiple workers, and bad `collate_fn` designs cause padded sequences to be twice the memory they need to be.

In [ ]:
%pip install -q torch numpy matplotlib

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, IterableDataset
import numpy as np
import matplotlib.pyplot as plt
import time
import os
import random

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")
print(f"torch: {torch.__version__}")
print(f"CPU count: {os.cpu_count()}")
print("Ready.")

## 1. Dataset __len__ and __getitem__ Contract

The map-style Dataset contract is minimal: implement `__len__` (returns total number of samples) and `__getitem__` (given an integer index, return the corresponding sample). The DataLoader handles batching, shuffling, and parallel loading.

Key design principle: do the heavy work in `__getitem__`, not `__init__`. Load the file, decode the image, apply the transform in `__getitem__` so workers can do it in parallel. Store only file paths and metadata in `__init__`.

In [ ]:
class SyntheticDataset(Dataset):
    def __init__(self, n_samples=1000, feature_dim=32, n_classes=5, seed=42):
        super().__init__()
        rng = np.random.RandomState(seed)
        # Store metadata, not all data (for illustration)
        self.n_samples   = n_samples
        self.feature_dim = feature_dim
        self.n_classes   = n_classes
        # In a real dataset you'd store file paths here
        self.labels = rng.randint(0, n_classes, size=n_samples)

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        # Simulate per-sample loading + transform
        rng = np.random.RandomState(idx)  # deterministic per idx
        x = torch.from_numpy(
            rng.randn(self.feature_dim).astype(np.float32)
        )
        y = int(self.labels[idx])
        return x, y

ds = SyntheticDataset()
print(f"Dataset length: {len(ds)}")
x_sample, y_sample = ds[0]
print(f"Sample shape: {x_sample.shape}, label: {y_sample}")

# Basic DataLoader
loader = DataLoader(ds, batch_size=32, shuffle=True)
batch_x, batch_y = next(iter(loader))
print(f"Batch shape: {batch_x.shape}, labels shape: {batch_y.shape}")

## 2. Map-style vs IterableDataset

Use **map-style** (default) when:
- Your dataset fits in memory or you can index into it by integer ID
- You need proper shuffling
- Data lives in files you can seek to by index

Use **IterableDataset** when:
- Data is a stream that cannot be indexed (live log files, network streams, infinite augmentation pipelines)
- Dataset is too large to enumerate (web-scale crawls)
- You are generating synthetic data on-the-fly

Critical caveat: with `num_workers > 0`, each worker gets a full copy of the IterableDataset. Without a `worker_init_fn` that shards the stream, you get duplicate data from every worker.

In [ ]:
class InfiniteNormalStream(IterableDataset):
    def __init__(self, feature_dim=8, n_classes=3, worker_aware=True):
        super().__init__()
        self.feature_dim  = feature_dim
        self.n_classes    = n_classes
        self.worker_aware = worker_aware

    def __iter__(self):
        worker_info = torch.utils.data.get_worker_info()
        if worker_info is not None and self.worker_aware:
            # Each worker gets a unique seed to generate different samples
            seed = worker_info.id * 1000 + int(torch.initial_seed()) % 1000
        else:
            seed = 42
        rng = np.random.RandomState(seed)
        while True:
            x = rng.randn(self.feature_dim).astype(np.float32)
            y = rng.randint(0, self.n_classes)
            yield torch.from_numpy(x), y

# Demonstrate: without worker awareness, all workers produce same sequence
ds_stream = InfiniteNormalStream(worker_aware=False)
loader_stream = DataLoader(ds_stream, batch_size=4, num_workers=2)
batch1 = next(iter(loader_stream))
batch2 = next(iter(loader_stream))
x1, x2 = batch1[0], batch2[0]
print(f"Worker-unaware: first elements equal? {torch.allclose(x1[0], x2[0])}")

# With worker awareness, each worker generates unique samples
ds_stream_aware = InfiniteNormalStream(worker_aware=True)
loader_aware = DataLoader(ds_stream_aware, batch_size=4, num_workers=2)
batch_a = next(iter(loader_aware))
batch_b = next(iter(loader_aware))
print(f"Worker-aware:   first elements equal? {torch.allclose(batch_a[0][0], batch_b[0][0])}")
print("(should be False -- each worker produces different samples)")

## 3. DataLoader Performance Knobs

The most impactful DataLoader parameters for GPU training performance:

- **num_workers**: number of parallel processes for data loading. Rule of thumb: 2-4x number of GPUs. Setting too high causes excessive memory use and process coordination overhead.
- **pin_memory=True**: pins CPU memory for faster host-to-device (H2D) transfer. Only meaningful when training on CUDA/MPS. Uses more RAM.
- **persistent_workers=True**: keeps worker processes alive between epochs. Avoids the ~1s overhead of spawning workers each epoch. Requires `num_workers > 0`.
- **prefetch_factor**: number of batches pre-fetched per worker. Default 2. Increase if workers are fast and GPU is idle waiting for data.

In [ ]:
import time

ds_perf = SyntheticDataset(n_samples=2000)

def benchmark_loader(loader, n_batches=50):
    it = iter(loader)
    # warmup
    for _ in range(3):
        next(it)
    start = time.perf_counter()
    for _ in range(n_batches):
        batch = next(it)
    elapsed = time.perf_counter() - start
    return elapsed

configs = [
    {"num_workers": 0, "pin_memory": False,  "persistent_workers": False, "label": "0 workers"},
    {"num_workers": 2, "pin_memory": False,  "persistent_workers": False, "label": "2 workers"},
    {"num_workers": 4, "pin_memory": False,  "persistent_workers": True,  "label": "4 workers + persistent"},
]

print("DataLoader benchmark (50 batches, batch_size=64):")
for cfg in configs:
    label = cfg.pop("label")
    loader_bench = DataLoader(ds_perf, batch_size=64, **cfg)
    t = benchmark_loader(loader_bench)
    # restore label for printing
    cfg["label"] = label
    print(f"  {label:<35s}: {t:.3f}s")

print("\nNote: absolute times vary by machine; relative ordering matters most.")
print("On GPU machines with fast disk, 4 workers + persistent typically saves 20-40% wallclock time.")

## 4. Custom collate_fn for Variable-Length Sequences

The default `collate_fn` stacks tensors along dimension 0, which requires all samples in a batch to have the same shape. For NLP tasks (variable-length token sequences) or audio, you must pad sequences to the same length within a batch and pass attention masks. A custom `collate_fn` handles this cleanly.

In [ ]:
from torch.nn.utils.rnn import pad_sequence

class VariableLengthDataset(Dataset):
    def __init__(self, n_samples=200, vocab_size=100, seed=42):
        super().__init__()
        rng = np.random.RandomState(seed)
        self.sequences = [
            torch.tensor(rng.randint(1, vocab_size, size=rng.randint(5, 30)), dtype=torch.long)
            for _ in range(n_samples)
        ]
        self.labels = torch.randint(0, 3, (n_samples,))

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

def pad_collate(batch):
    seqs, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs], dtype=torch.long)
    # pad_sequence stacks along batch dimension, padding to max length in batch
    padded = pad_sequence(seqs, batch_first=True, padding_value=0)
    # attention mask: 1 where real token, 0 where padding
    mask = (padded != 0).long()
    labels = torch.stack(labels)
    return {"input_ids": padded, "attention_mask": mask, "lengths": lengths, "labels": labels}

vl_ds = VariableLengthDataset()
vl_loader = DataLoader(vl_ds, batch_size=8, collate_fn=pad_collate, shuffle=True)
batch = next(iter(vl_loader))
print(f"input_ids shape:     {batch['input_ids'].shape}")
print(f"attention_mask shape:{batch['attention_mask'].shape}")
print(f"lengths:             {batch['lengths'].tolist()}")
print(f"labels shape:        {batch['labels'].shape}")
print(f"padding fraction:    {(batch['input_ids'] == 0).float().mean():.2f}")

## 5. pin_memory + non_blocking for Async H2D Transfer

The default `.to(device)` call is synchronous -- the CPU waits for the transfer to complete before continuing. With `pin_memory=True` in the DataLoader and `non_blocking=True` in `.to()`, the H2D transfer is asynchronous: it runs in the background while the CPU prepares the next batch. This overlaps data transfer with computation, improving GPU utilisation.

In [ ]:
# pin_memory + non_blocking pattern
ds_pin = SyntheticDataset(n_samples=500)
loader_pin = DataLoader(ds_pin, batch_size=64, num_workers=2, pin_memory=(device != "cpu"))

# The correct async transfer pattern
print("Async H2D transfer pattern:")
for i, (x_batch, y_batch) in enumerate(loader_pin):
    # non_blocking=True: initiates transfer, does not wait
    x_gpu = x_batch.to(device, non_blocking=True)
    y_gpu = y_batch.to(device, non_blocking=True)
    # GPU ops will automatically sync before using these tensors
    # Meanwhile CPU can pre-load next batch
    if i == 0:
        print(f"  x_batch device: {x_batch.device} (CPU, pinned: {x_batch.is_pinned()})")
        print(f"  x_gpu device:   {x_gpu.device}")
    if i >= 2:
        break

print("\nNote: is_pinned() returns False on MPS; pin_memory is a CUDA feature.")
print("On CUDA machines, pin_memory=True + non_blocking=True gives ~5-15% speedup.")

## 6. worker_init_fn for Reproducible Seeding

When using `num_workers > 0`, each worker inherits the same random state from the parent process. Two consequences:
1. Multiple workers generate the same random augmentations (correlated augmentation within a batch)
2. Restarting training from a checkpoint may not reproduce the exact data order/augmentation

The fix: provide a `worker_init_fn` that seeds each worker independently based on the worker ID and the global seed.

In [ ]:
def seed_worker(worker_id):
    # Each worker gets a unique, deterministic seed
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(42)

ds_seed = SyntheticDataset()
loader_seeded = DataLoader(
    ds_seed,
    batch_size=32,
    num_workers=2,
    worker_init_fn=seed_worker,
    generator=g,   # controls DataLoader's own shuffle order
    shuffle=True,
)

# verify reproducibility
def collect_indices(loader, n=2):
    indices = []
    for i, (x, y) in enumerate(loader):
        indices.extend(y[:4].tolist())
        if i >= n:
            break
    return indices

# Two iterations with same seed should give same order
g.manual_seed(42)
first_run  = collect_indices(loader_seeded)
g.manual_seed(42)
second_run = collect_indices(loader_seeded)
print(f"Reproducible label order: {first_run == second_run}")
print(f"First run  labels: {first_run}")
print(f"Second run labels: {second_run}")

## Summary

### Key Concepts

| Concept | Key Point |
|---------|----------|
| __getitem__ contract | Heavy work here, not __init__; workers call it in parallel |
| IterableDataset | For streams; must shard across workers manually |
| num_workers | 2-4x GPU count; 0 means synchronous main-process loading |
| pin_memory + non_blocking | Overlaps H2D transfer with computation; CUDA only |
| persistent_workers | Keeps worker procs alive between epochs; saves spawn overhead |
| custom collate_fn | Required for variable-length sequences; pad + mask |
| worker_init_fn | Unique seed per worker; required for reproducible augmentation |

### Common Pitfalls
- Opening files in `__init__` instead of `__getitem__` -- file handles are not fork-safe across workers
- IterableDataset without worker sharding -- every worker produces every sample (N x duplication)
- `pin_memory=True` on CPU-only machines -- does nothing, wastes RAM
- Not setting `generator` seed when you need reproducible shuffling across runs
- Forgetting `persistent_workers=True` -- worker spawn overhead can be 5-10% of epoch time

### Next: Notebook 06 -- Mixed Precision Training